### Dans utils.py :

In [ ]:
#NEW
def get_moments_of_inertia_for_size(self, vectors=False): #from ASE but with mass modification
    '''
    Get the moments of inertia along the principal axes with
    mass normalisation.
    The three principal moments of inertia are computed from the
    eigenvalues of the symmetric inertial tensor. Periodic boundary
    conditions are ignored. 
    Units of the moments of inertia are angstrom**2.
    '''
    com = self.get_center_of_mass()
    positions = self.get_positions()
    #number_atoms=len(positions)
    positions -= com  # translate center of mass to origin
    masses = self.get_masses()

    # Initialize elements of the inertial tensor
    I11 = I22 = I33 = I12 = I13 = I23 = 0.0
    for i in range(len(self)):
        x, y, z = positions[i]
        m = 1
        I11 += m * (y ** 2 + z ** 2)
        I22 += m * (x ** 2 + z ** 2)
        I33 += m * (x ** 2 + y ** 2)
        I12 += -m * x * y
        I13 += -m * x * z
        I23 += -m * y * z

    Itensor = np.array([[I11, I12, I13],
                        [I12, I22, I23],
                        [I13, I23, I33]])

    evals, evecs = np.linalg.eigh(Itensor) #valeurs propes de la matrice 
    if vectors:
        return evals, evecs.transpose()
    else:
        return evals


def moi_size(model: Atoms, # normalized moment of inertia with masses=1
        noOutput: bool=False,
       ):
    '''
    Returns the moments of inertia along the principal axes with mass normalization to get acces to size informations
    '''

    model.moi_size_all = get_moments_of_inertia_for_size(model)
    positions = model.get_positions()
    number_atoms=len(positions)
    model.moi_size = model.moi_size_all/number_atoms
    if not noOutput: print(f"Moments of inertia with mass=1/ M = {model.moi_size[0]:.2f} {model.moi_size[1]:.2f} {model.moi_size[2]:.2f} Å2")
    return [model.moi_size[0],model.moi_size[1],model.moi_size[2]]

### Dans crystalNPs.py :

In [ ]:
 def propPostMake(self,skipSymmetryAnalyzis, thresholdCoreSurface, noOutput): #accès aux faces voisines etc
        self.dim=[0,0,0]
        pyNMBu.moi(self.NP, noOutput) # usual MOI
        moiMsize=pyNMBu.moi_size(self.NP, noOutput)   # MOI mass normalized (m of each atoms=1)

     
        # find the size using the MOI mass normalized
        if self.shape == 'ellipsoid': # https://scienceworld.wolfram.com/physics/MomentofInertiaEllipsoid.html
            self.dim[0] = np.sqrt((5 * moiMsize[1] + 5 * moiMsize[2] - 5 * moiMsize[0]) / 2)
            self.dim[1] = np.sqrt((5 * moiMsize[0] + 5 * moiMsize[2] - 5 * moiMsize[1]) / 2)
            self.dim[2] = np.sqrt((5 * moiMsize[0] + 5 * moiMsize[1] - 5 * moiMsize[2]) / 2)
            if not noOutput:
                print(f"Radius of the ellipsoid  { self.dim[0]* 0.1:.2f}  { self.dim[1] * 0.1:.2f}  { self.dim[2] * 0.1:.2f} nm")
        if self.shape == 'sphere': #wikipedia
            self.dim[0] = 2 * np.sqrt(5 / 2 * moiMsize[0])  # same formula cause Ix, Iy, Iz are equals
            self.dim[1] = 2 * np.sqrt(5 / 2 * moiMsize[0])
            self.dim[2] = 2 * np.sqrt(5 / 2 * moiMsize[0])
            if not noOutput:
                print(f"Diameter of the sphere = {self.dim[0] * 0.1:.2f}  {self.dim[1] * 0.1:.2f}  {self.dim[2] * 0.1:.2f} nm")
        if self.shape == 'cylinder': #wikipedia
            self.dim[0] = np.sqrt(12 * moiMsize[1] - 6 * moiMsize[0]) #longest distance
            self.dim[1] = 2 * np.sqrt(2 * moiMsize[0]) 
            self.dim[2] = 2 * np.sqrt(2 * moiMsize[0])
            if not noOutput:
                print(f"Size of the cylinder = {self.dim[0] * 0.1:.2f} {self.dim[1] * 0.1:.2f} {self.dim[2] * 0.1:.2f} nm")
        if self.shape == 'parallepiped': #wikipedia
            self.dim[0] = np.sqrt((6 * moiMsize[1] + moiMsize[2] - moiMsize[0]) / 2) #longest distance
            self.dim[1] = np.sqrt((6 * moiMsize[0] + moiMsize[2] - moiMsize[1]) / 2) # 2nd longest distance
            self.dim[2] = np.sqrt((6 * moiMsize[0] + moiMsize[1] - moiMsize[2]) / 2) # 3rd longest distance
            if not noOutput:
                print(f"Size of the parallepiped=  {self.dim[0] * 0.1:.2f}  {self.dim[1] * 0.1:.2f}  {self.dim[2] * 0.1:.2f} nm")

        if self.shape == 'wire': #wikipedia and me (can be calculated with basic formulas)
            if self.nRotWire==4 :
                self.dim[0]=np.sqrt(12*moiMsize[1]-6*moiMsize[0]) #longest distance
                self.dim[1]=np.sqrt(6*moiMsize[0])
                self.dim[2]=np.sqrt(6*moiMsize[0])
                if not noOutput:
                    print(f"Size of the wire=  {self.dim[0] * 0.1:.2f}  {self.dim[1] * 0.1:.2f}  {self.dim[2] * 0.1:.2f} nm")
            if self.nRotWire==6 :
                self.dim[0]=np.sqrt(12*moiMsize[1]-6*moiMsize[0]) #longest distance
                self.dim[1]=2*np.sqrt(2*moiMsize[0])
                self.dim[2]=2*np.sqrt(2*moiMsize[0])
                if not noOutput:
                    print(f"Size of the wire=  {self.dim[0] * 0.1:.2f}  {self.dim[1] * 0.1:.2f}  {self.dim[2] * 0.1:.2f} nm")
                